# Notebook 7 / 8 — The Workflow as a LangGraph

> **Series.** This is the seventh of eight notebooks.
> 1. *Basics* — eight foundational logics
> 2. *Advanced* — fourteen rarer logics
> 3. *Synthesis* — cross-logic benchmarks and conclusions
> 4. *Applications* — ten general-purpose domains
> 5. *Language* — non-classical logics in NLP tasks
> 6. *Workflow* — an end-to-end pipeline composing the best logics, plain Python
> 7. **LangGraph** *(this notebook)* — the same pipeline rebuilt as a LangGraph state machine
> 8. *Experimental composition* — probing what happens when two non-classical logics overlap on the same linguistic phenomenon

## What this notebook is

Notebook 6 built a 9-stage agent pipeline (defaults → intuitionistic retrieval → DS sense disambiguation → possibilistic+LP fusion → subjective opinion aggregation → SDL norm check → DEL privacy gate → LTL protocol check → bilattice decision) as a flat sequence of Python functions.

This notebook **promotes that pipeline to a graph**. The advantages of using LangGraph specifically:

1. **Conditional routing** — early refusal (norm violation, privacy leak, no witness) becomes a labelled edge to `END` instead of a dead branch in `decide()`.
2. **Typed state** — the `ClaimState` is the single object the graph carries; every node only writes the slice it owns.
3. **Visible topology** — the graph can be drawn (`graph.get_graph().draw_mermaid()`), making the disjoint-ownership principle from notebook 6 visually explicit.
4. **Reusable subgraphs** — the *evidence cluster* (retrieve → DS → possibilistic+LP → subjective) can be lifted into its own subgraph for reuse in other agents.
5. **Free instrumentation** — checkpointers, streaming, and step-by-step inspection come for free.

## The graph at a glance

```
                              ┌────────────┐
                              │   START    │
                              └─────┬──────┘
                                    ▼
                          ┌─────────────────┐
                          │ parse_intent    │  (default logic)
                          └────────┬────────┘
                                   ▼
                          ┌─────────────────┐
                          │ check_norms     │  (SDL)
                          └────────┬────────┘
                          forbidden│ │ ok
                  ┌────────────────┘ └────────────┐
                  ▼                               ▼
             ┌─────────┐                 ┌──────────────────┐
             │  REFUSE │                 │ retrieve_witness │  (intuitionistic)
             └────┬────┘                 └────────┬─────────┘
                  │              no witness │     │ has witness
                  │            ┌────────────┘     ▼
                  │            ▼            ┌──────────────┐
                  │       ┌─────────┐       │ disambiguate │  (Dempster–Shafer)
                  │       │ ESCALATE│       └──────┬───────┘
                  │       └────┬────┘              ▼
                  │            │           ┌──────────────┐
                  │            │           │ fuse_evidence│  (possibilistic + LP)
                  │            │           └──────┬───────┘
                  │            │                  ▼
                  │            │           ┌──────────────────┐
                  │            │           │ aggregate_opinion│  (subjective)
                  │            │           └──────┬───────────┘
                  │            │                  ▼
                  │            │           ┌──────────────┐
                  │            │           │ check_privacy│  (DEL gate)
                  │            │           └──────┬───────┘
                  │            │     leak │       │ ok
                  │            │       ┌──┘       ▼
                  │            │       │   ┌──────────────┐
                  │            │       │   │ check_protocol│ (LTL)
                  │            │       │   └──────┬───────┘
                  │            │       │          ▼
                  │            │       │   ┌──────────────┐
                  │            │       │   │   decide     │  (bilattice)
                  │            │       │   └──────┬───────┘
                  │            │       │          ▼
                  └────────────┴───────┴──────► END
```

## Setup

We use `langgraph` (and `typing_extensions` for `TypedDict`). If LangGraph is not installed, the next cell installs it; the rest of the notebook is pure Python plus the `StateGraph` API.

In [1]:
# Install LangGraph if not present.
try:
    import langgraph
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langgraph"])
    import langgraph
print("langgraph version:", langgraph.__version__ if hasattr(langgraph, "__version__") else "installed")

langgraph version: installed


In [2]:
from typing import Dict, List, Set, Tuple, Optional, FrozenSet, Literal
from typing_extensions import TypedDict, NotRequired
from dataclasses import dataclass, field
from functools import reduce

from langgraph.graph import StateGraph, START, END

## A. Shared state

LangGraph's `StateGraph` works against a `TypedDict`. Each node returns a *partial* update; LangGraph merges it into the running state. We mark every field `NotRequired` so nodes can write whichever slice they own without restating the rest.

In [3]:
# --- helper data types ---
Tv = frozenset({"T"}); Fv = frozenset({"F"}); Bv = frozenset({"T","F"}); Nv = frozenset()

@dataclass
class Opinion:
    b: float = 0.0
    d: float = 0.0
    u: float = 1.0
    a: float = 0.5
    def proj(self): return self.b + self.a*self.u

# Serialisation helpers — frozensets/sets cannot be JSON keys (LangSmith tracer).
def _lp_str(v: frozenset) -> str:
    return ",".join(sorted(v))          # "" = N, "F" = F, "T" = T, "F,T" = B

def _lp_parse(s: str) -> frozenset:
    return frozenset(s.split(",")) if s else frozenset()

def _senses_str(m: dict) -> dict:      # {frozenset → float} → {str → float}
    return {",".join(sorted(k)): v for k, v in m.items()}

# --- the graph state ---
class ClaimState(TypedDict, total=False):
    # input
    question: str
    target_atom: str
    cues: List[str]
    private_passages: List[str]     # list — JSON-serializable
    contradicts: bool
    # filled in by nodes
    intent: List[str]               # list — JSON-serializable
    forbidden: bool
    obligations: List[str]          # list — JSON-serializable
    witnesses: List[str]
    senses: Dict[str, float]        # serialised: ",".join(sorted(frozenset))
    necessity: float
    lp_value: str                   # serialised LP value: "", "T", "F", or "F,T"
    opinion: Opinion
    privacy_violated: bool
    protocol_ok: bool
    # output
    action: str
    trace: List[str]

## B. The knowledge fixtures

Same toy KB as notebook 6 — kept here so the notebook is independent.

In [4]:
KB_PASSAGES = {
    "para_1": ("ibuprofen typical adult dose is 200-400 mg every 4-6 hours",
               {"ibuprofen_dose"}),
    "para_2": ("high blood pressure is defined as systolic >= 130 mmHg",
               {"hypertension_threshold"}),
    "para_3": ("depression screening uses the PHQ-9 questionnaire",
               {"depression_screening"}),
    "para_4": ("acetaminophen and ibuprofen can be alternated under clinician supervision",
               {"ibuprofen_combo", "clinician_required"}),
}
PASSAGE_RELIABILITY = {"para_1":0.95, "para_2":0.90, "para_3":0.85, "para_4":0.92}

fs  = lambda *xs: frozenset(xs)
fss = lambda *xs: ",".join(sorted(xs))   # string key version

SENSE_PRIORS = {
    "pressure":   {fss("blood_pressure","atmospheric","social"): 1.0},
    "depression": {fss("clinical","economic","geographic"): 1.0},
}
CONTEXT_CUES = {
    "blood":    {fss("blood_pressure"): 0.8, fss("blood_pressure","social"): 0.2},
    "weather":  {fss("atmospheric"): 0.9, fss("atmospheric","social"): 0.1},
    "clinical": {fss("clinical"): 0.85, fss("clinical","economic"): 0.15},
    "market":   {fss("economic"): 0.9, fss("economic","geographic"): 0.1},
}

@dataclass
class Default:
    pre: str; justification: str; conclusion: str

DEFAULTS = [Default("medical_question","safe_advice","wants_safe_advice")]
OBLIGATION_TRIGGERS = {"medical_question":"add_disclaimer"}

## C. Graph nodes — one logic per function

Each node receives the full `ClaimState` and returns the **partial update** corresponding to its own slice. The discipline from notebook 6 ("each logic owns one slice of state") is now enforced by the type system: a node literally cannot touch fields it does not name.

In [5]:
# --- 1. parse_intent — default logic ---------------------------------------
def parse_intent(state: ClaimState) -> ClaimState:
    q = state["question"].lower()
    facts = []
    if any(k in q for k in ["dose","symptom","medication","pressure","depression"]):
        facts.append("medical_question")
    if "just curious" in q:                       facts.append("not_safe_advice")
    if "my friend" in q or "someone else" in q:   facts.append("third_party_question")
    # Apply defaults to closure
    out = list(facts); changed = True
    while changed:
        changed = False
        for d in DEFAULTS:
            if d.pre in out and ("not_"+d.justification) not in out and d.conclusion not in out:
                out.append(d.conclusion); changed = True
    return {"intent": out, "trace": state.get("trace", []) + ["parse_intent"]}

In [6]:
# --- 2. check_norms — SDL --------------------------------------------------
def check_norms(state: ClaimState) -> ClaimState:
    intent = state["intent"]
    forbidden = ("third_party_question" in intent and "medical_question" in intent)
    obligations = [OBLIGATION_TRIGGERS[k] for k in intent if k in OBLIGATION_TRIGGERS]
    return {
        "forbidden": forbidden,
        "obligations": obligations,
        "trace": state.get("trace", []) + ["check_norms"],
    }

In [7]:
# --- 3. retrieve_witness — intuitionistic ----------------------------------
def retrieve_witness(state: ClaimState) -> ClaimState:
    target = state["target_atom"]
    witnesses = [pid for pid,(_,atoms) in KB_PASSAGES.items() if target in atoms]
    # The 'clinician_required' tag escalates retrieved evidence to forbidden.
    forbidden = state.get("forbidden", False) or any(
        "clinician_required" in KB_PASSAGES[p][1] for p in witnesses
    )
    return {
        "witnesses": witnesses,
        "forbidden": forbidden,
        "trace": state.get("trace", []) + ["retrieve_witness"],
    }

In [8]:
# --- 4. disambiguate — Dempster-Shafer -------------------------------------
def _dempster(m1, m2):
    out, K = {}, 0.0
    for A,pA in m1.items():
        for B,pB in m2.items():
            inter = A & B
            if not inter: K += pA*pB
            else: out[inter] = out.get(inter,0) + pA*pB
    if K >= 1.0: return out
    return {s:v/(1-K) for s,v in out.items()}

def disambiguate(state: ClaimState) -> ClaimState:
    q = state["question"].lower()
    term = "pressure" if "pressure" in q else "depression" if "depression" in q else "unknown"
    mass = SENSE_PRIORS.get(term, {fs("unknown"):1.0})
    for cue in state.get("cues", []):
        if cue in CONTEXT_CUES:
            mass = _dempster(mass, CONTEXT_CUES[cue])
    return {"senses": _senses_str(mass), "trace": state.get("trace", []) + ["disambiguate"]}

In [9]:
# --- 5. fuse_evidence — possibilistic + LP ---------------------------------
def fuse_evidence(state: ClaimState) -> ClaimState:
    witnesses = state.get("witnesses", [])
    contradicts = state.get("contradicts", False)
    if not witnesses:
        return {"necessity": 0.0, "lp_value": _lp_str(Nv),
                "trace": state.get("trace", []) + ["fuse_evidence"]}
    necessity = max(PASSAGE_RELIABILITY[p] for p in witnesses)
    lp = Tv
    if contradicts: lp = frozenset(lp | Fv)        # becomes B
    return {"necessity": necessity, "lp_value": _lp_str(lp),
            "trace": state.get("trace", []) + ["fuse_evidence"]}

In [10]:
# --- 6. aggregate_opinion — subjective logic --------------------------------
def _fuse_op(o1: Opinion, o2: Opinion) -> Opinion:
    if o1.u == 0 and o2.u == 0:
        return Opinion((o1.b+o2.b)/2, (o1.d+o2.d)/2, 0.0, o1.a)
    den = o1.u + o2.u - o1.u*o2.u
    return Opinion(
        (o1.b*o2.u + o2.b*o1.u)/den,
        (o1.d*o2.u + o2.d*o1.u)/den,
        (o1.u*o2.u)/den,
        o1.a,
    )

def aggregate_opinion(state: ClaimState) -> ClaimState:
    witnesses = state.get("witnesses", [])
    contradicts = state.get("contradicts", False)
    if not witnesses:
        return {"opinion": Opinion(0.0, 0.0, 1.0),
                "trace": state.get("trace", []) + ["aggregate_opinion"]}
    ops = [Opinion(b=PASSAGE_RELIABILITY[p], d=0.0, u=1-PASSAGE_RELIABILITY[p]) for p in witnesses]
    if contradicts:
        ops.append(Opinion(b=0.0, d=0.85, u=0.15))
    return {"opinion": reduce(_fuse_op, ops),
            "trace": state.get("trace", []) + ["aggregate_opinion"]}

In [11]:
# --- 7. check_privacy — DEL gate -------------------------------------------
def check_privacy(state: ClaimState) -> ClaimState:
    private = state.get("private_passages", set())
    leak = any(p in private for p in state.get("witnesses", []))
    return {"privacy_violated": leak,
            "trace": state.get("trace", []) + ["check_privacy"]}

In [12]:
# --- 8. check_protocol — LTL ------------------------------------------------
def _ltl(f, trace, i=0):
    n = len(trace)
    if i >= n: return False
    op = f[0]
    if op=="atom": return f[1] in trace[i]
    if op=="not":  return not _ltl(f[1],trace,i)
    if op=="and":  return _ltl(f[1],trace,i) and _ltl(f[2],trace,i)
    if op=="or":   return _ltl(f[1],trace,i) or  _ltl(f[2],trace,i)
    if op=="imp":  return (not _ltl(f[1],trace,i)) or _ltl(f[2],trace,i)
    if op=="F":    return any(_ltl(f[1],trace,j) for j in range(i,n))
    if op=="G":    return all(_ltl(f[1],trace,j) for j in range(i,n))
    raise ValueError(op)

PROTOCOL = ("G",("imp",("atom","request"),
                       ("F",("or",("atom","answer"),("atom","escalate")))))

def check_protocol(state: ClaimState) -> ClaimState:
    # Synthesise the obligation: the conversation contains a request and must terminate.
    pseudo_trace = [{"request"}, {"escalate"}]   # provisional — overridden by `decide`
    return {"protocol_ok": _ltl(PROTOCOL, pseudo_trace),
            "trace": state.get("trace", []) + ["check_protocol"]}

In [13]:
# --- 9. decide — bilattice collapse ----------------------------------------
def decide(state: ClaimState) -> ClaimState:
    lp = _lp_parse(state.get("lp_value", ""))   # deserialise back to frozenset
    op = state.get("opinion", Opinion())
    if lp == Bv:                          belnap = "B"
    elif lp == Tv and op.proj() > 0.7:    belnap = "T"
    elif lp == Fv:                        belnap = "F"
    else:                                  belnap = "N"

    if belnap == "T":
        suffix = " + DISCLAIMER" if "add_disclaimer" in state.get("obligations", set()) else ""
        action = f"ANSWER (cite {state.get('witnesses', [])}){suffix}"
    elif belnap == "B":   action = "ESCALATE (sources disagree)"
    elif belnap == "N":   action = "ESCALATE (insufficient evidence)"
    else:                  action = "REFUSE (negative evidence dominates)"
    return {"action": action,
            "trace": state.get("trace", []) + ["decide"]}

# Terminal nodes for early refusal / escalation
def refuse_node(state: ClaimState) -> ClaimState:
    reason = ("forbidden by norms"   if state.get("forbidden")        else
              "would leak private info" if state.get("privacy_violated") else
              "no constructive witness")
    return {"action": f"REFUSE ({reason})",
            "trace": state.get("trace", []) + ["refuse"]}

def escalate_no_witness(state: ClaimState) -> ClaimState:
    return {"action": "ESCALATE (no retrieved witness — ask clarifying Q)",
            "trace": state.get("trace", []) + ["escalate_no_witness"]}

## D. Conditional edges

These are the routing functions LangGraph calls after each node to decide where the state goes next.

In [14]:
def route_after_norms(state: ClaimState) -> Literal["retrieve_witness", "refuse"]:
    return "refuse" if state.get("forbidden") else "retrieve_witness"

def route_after_retrieve(state: ClaimState) -> Literal["disambiguate", "escalate_no_witness", "refuse"]:
    if state.get("forbidden"):                # clinician_required appeared after retrieval
        return "refuse"
    if not state.get("witnesses"):
        return "escalate_no_witness"
    return "disambiguate"

def route_after_privacy(state: ClaimState) -> Literal["check_protocol", "refuse"]:
    return "refuse" if state.get("privacy_violated") else "check_protocol"

## E. Build the graph

In [15]:
builder = StateGraph(ClaimState)

# Register nodes
builder.add_node("parse_intent",         parse_intent)
builder.add_node("check_norms",          check_norms)
builder.add_node("retrieve_witness",     retrieve_witness)
builder.add_node("disambiguate",         disambiguate)
builder.add_node("fuse_evidence",        fuse_evidence)
builder.add_node("aggregate_opinion",    aggregate_opinion)
builder.add_node("check_privacy",        check_privacy)
builder.add_node("check_protocol",       check_protocol)
builder.add_node("decide",               decide)
builder.add_node("refuse",               refuse_node)
builder.add_node("escalate_no_witness",  escalate_no_witness)

# Linear edges first
builder.add_edge(START, "parse_intent")
builder.add_edge("parse_intent", "check_norms")

# Conditional: norms gate
builder.add_conditional_edges("check_norms", route_after_norms, {
    "retrieve_witness": "retrieve_witness",
    "refuse":           "refuse",
})

# Conditional: retrieval gate
builder.add_conditional_edges("retrieve_witness", route_after_retrieve, {
    "disambiguate":         "disambiguate",
    "escalate_no_witness":  "escalate_no_witness",
    "refuse":               "refuse",
})

# Linear evidence cluster
builder.add_edge("disambiguate",      "fuse_evidence")
builder.add_edge("fuse_evidence",     "aggregate_opinion")
builder.add_edge("aggregate_opinion", "check_privacy")

# Conditional: privacy gate
builder.add_conditional_edges("check_privacy", route_after_privacy, {
    "check_protocol": "check_protocol",
    "refuse":         "refuse",
})

# Tail
builder.add_edge("check_protocol", "decide")
builder.add_edge("decide", END)
builder.add_edge("refuse", END)
builder.add_edge("escalate_no_witness", END)

graph = builder.compile()
print("compiled graph with", len(graph.get_graph().nodes), "nodes")

compiled graph with 13 nodes


### Visualise the topology

LangGraph can render the compiled graph as Mermaid. If you are running this in Jupyter with a Mermaid-aware renderer, the cell below will draw the diagram; otherwise it just prints the source.

In [16]:
try:
    mermaid = graph.get_graph().draw_mermaid()
    print(mermaid)
except Exception as e:
    print("Could not render Mermaid:", e)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse_intent(parse_intent)
	check_norms(check_norms)
	retrieve_witness(retrieve_witness)
	disambiguate(disambiguate)
	fuse_evidence(fuse_evidence)
	aggregate_opinion(aggregate_opinion)
	check_privacy(check_privacy)
	check_protocol(check_protocol)
	decide(decide)
	refuse(refuse)
	escalate_no_witness(escalate_no_witness)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse_intent;
	aggregate_opinion --> check_privacy;
	check_norms -.-> refuse;
	check_norms -.-> retrieve_witness;
	check_privacy -.-> check_protocol;
	check_privacy -.-> refuse;
	check_protocol --> decide;
	disambiguate --> fuse_evidence;
	fuse_evidence --> aggregate_opinion;
	parse_intent --> check_norms;
	retrieve_witness -.-> disambiguate;
	retrieve_witness -.-> escalate_no_witness;
	retrieve_witness -.-> refuse;
	decide --> __end__;
	escalate_no_witness --> __end__;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-h

In [17]:
print(type(list(CONTEXT_CUES["clinical"].keys())[0]))

<class 'str'>


## F. Run the four scenarios from notebook 6

Each invocation builds an initial `ClaimState`, calls `graph.invoke(...)`, and prints the resulting trace + final action.

In [18]:
def run(label, **kwargs):
    print("\n" + "═"*64)
    print("  " + label)
    print("═"*64)
    initial: ClaimState = {
        "question":          kwargs["question"],
        "target_atom":       kwargs["target_atom"],
        "cues":              kwargs.get("cues", []),
        "private_passages":  list(kwargs.get("private_passages", [])),  # set → list
        "contradicts":       kwargs.get("contradicts", False),
        "trace":             [],
    }
    final = graph.invoke(initial)
    print("Q:", final["question"])
    print("trace:", " → ".join(final["trace"]))
    print("witnesses:", final.get("witnesses", []))
    print("necessity:", final.get("necessity", 0.0),
          " lp:", set(final.get("lp_value", "").split(",")) if final.get("lp_value") else set(),
          " forbidden:", final.get("forbidden", False),
          " privacy_violated:", final.get("privacy_violated", False))
    print("ACTION →", final["action"])
    return final

_ = run("RUN 1 — clean medical question",
        question="What is the typical adult dose of ibuprofen?",
        target_atom="ibuprofen_dose",
        cues=["clinical"])


════════════════════════════════════════════════════════════════
  RUN 1 — clean medical question
════════════════════════════════════════════════════════════════


TypeError: unsupported operand type(s) for &: 'frozenset' and 'str'

In [ ]:
_ = run("RUN 2 — contradictory evidence",
        question="Are blood pressure readings above 130 already hypertension?",
        target_atom="hypertension_threshold",
        cues=["blood","clinical"],
        contradicts=True)


════════════════════════════════════════════════════════════════
  RUN 2 — contradictory evidence
════════════════════════════════════════════════════════════════
Q: Are blood pressure readings above 130 already hypertension?
trace: parse_intent → check_norms → retrieve_witness → disambiguate → fuse_evidence → aggregate_opinion → check_privacy → check_protocol → decide
witnesses: ['para_2']
necessity: 0.9  lp: {'T', 'F'}  forbidden: False  privacy_violated: False
ACTION → ESCALATE (sources disagree)


In [ ]:
_ = run("RUN 3 — forbidden third-party dosing question",
        question="My friend wants to know how much acetaminophen to mix with ibuprofen.",
        target_atom="ibuprofen_combo",
        cues=["clinical"])


════════════════════════════════════════════════════════════════
  RUN 3 — forbidden third-party dosing question
════════════════════════════════════════════════════════════════
Q: My friend wants to know how much acetaminophen to mix with ibuprofen.
trace: parse_intent → check_norms → retrieve_witness → refuse
witnesses: ['para_4']
necessity: 0.0  lp: set()  forbidden: True  privacy_violated: False
ACTION → REFUSE (forbidden by norms)


In [ ]:
print("Streaming RUN 4 with full state probe:\n")

def _find_bad(obj, path=""):
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if isinstance(k, (frozenset, set)):
                found.append(f"BAD KEY at {path}: {type(k).__name__}={k!r}")
            found += _find_bad(v, path + f"[{k!r}]")
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            found += _find_bad(v, path + f"[{i}]")
    elif isinstance(obj, (set, frozenset)):
        found.append(f"BAD VALUE at {path}: {type(obj).__name__}={obj!r}")
    return found

initial: ClaimState = {
    "question":         "How is depression usually screened?",
    "target_atom":      "depression_screening",
    "cues":             ["clinical"],
    "private_passages": ["para_3"],
    "contradicts":      False,
    "trace":            [],
}
for step in graph.stream(initial):
    for node, update in step.items():
        hits = _find_bad(update)
        marker = " *** BAD ***" if hits else ""
        print(f"  • {node:20s}{marker}")
        for h in hits:
            print(f"      {h}")

Streaming RUN 4 with full state probe:



NameError: name 'graph' is not defined

## G. Streaming the run, step by step

One of the things you get for free with LangGraph is **per-step streaming**. The same invocation can be observed as a sequence of node updates, which is exactly the audit trail the synthesis notebook (#3) wished for.

In [ ]:
print("Streaming RUN 2 (contradictory evidence):\n")
initial: ClaimState = {
    "question":     "Are blood pressure readings above 130 already hypertension?",
    "target_atom":  "hypertension_threshold",
    "cues":         ["blood","clinical"],
    "contradicts":  True,
    "trace":        [],
}
for step in graph.stream(initial):
    for node, update in step.items():
        new_keys = [k for k in update.keys() if k != "trace"]
        print(f"  • {node:20s}  wrote → {new_keys}")

Streaming RUN 2 (contradictory evidence):

  • parse_intent          wrote → ['intent']
  • check_norms           wrote → ['forbidden', 'obligations']
  • retrieve_witness      wrote → ['witnesses', 'forbidden']
  • disambiguate          wrote → ['senses']
  • fuse_evidence         wrote → ['necessity', 'lp_value']
  • aggregate_opinion     wrote → ['opinion']
  • check_privacy         wrote → ['privacy_violated']
  • check_protocol        wrote → ['protocol_ok']
  • decide                wrote → ['action']


## H. What changed by going from notebook 6 to LangGraph?

| Concern | Notebook 6 (plain Python) | Notebook 7 (LangGraph) |
|---|---|---|
| Control flow | hand-written `if` ladder in `pipeline()` | declarative `add_conditional_edges` |
| Early refusal | embedded in `decide()` after the fact | first-class edges to terminal `refuse` / `escalate` nodes |
| State discipline | dataclass + manual care | typed `ClaimState`; nodes return *only* their slice |
| Topology | implicit | explicit, drawable as a graph |
| Step-level audit | print statements | `graph.stream()` yields per-node updates |
| Reuse | copy/paste a function | lift a subgraph |

**The non-classical logics did not change.** What changed is *how the workflow is wired*. LangGraph turns the principles from notebook 6 — *carry the gradient*, *disjoint ownership*, *refusal beats incorrect commitment* — from coding conventions into structural invariants of the graph.

## I. Variations to try

- **Add a checkpointer** (`MemorySaver`) so each conversation has a thread id and the agent can resume after escalation.
- **Wrap the evidence cluster** (`disambiguate → fuse → aggregate`) as a *subgraph* and reuse it in a parallel agent that handles non-medical questions.
- **Replace `check_protocol` with a CTL model checker** when the conversation can branch (clarifying follow-ups), so the protocol property is verified over the entire reachable tree, not a single trace.
- **Plug in an LLM** at `parse_intent` to extract the structured intent fields, and at the very end to render the cited answer in natural language. Crucially, the *logic* nodes between them stay deterministic — the LLM never touches the gradient.
- **Send conflicting evidence to a second graph** that runs *AGM revision* over a plausibility ranking, then loops back to `aggregate_opinion`.

## Closing thought

Across seven notebooks the lesson narrows to a single sentence:

> **Pick the smallest non-classical fragment that names the detail your system can't afford to mis-model — then wire those fragments into a graph where every edge is a decision you would actually want to audit.**

Logic gives you the vocabulary; LangGraph gives you the invariant that nothing collapses to a boolean before it has to.